In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
import joblib
import os

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading cleaned data...")
df = pd.read_csv('D:\\Desktop\\BootCamp_Hackathon\\data\\processed\\cleaned_appointments.csv')

# Drop irrelevant columns (Prevents Data Leakage)
cols_to_drop = ['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay', 'Neighbourhood']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ==========================================
# 2. DEFINE FEATURES AND TARGET
# ==========================================
# Added errors='ignore' just in case Specialty was already dropped
X = df.drop(columns=['NoShow', 'Specialty'], errors='ignore') 
y = df['NoShow']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training on {len(X_train)} rows, Testing on {len(X_test)} rows...\n")

# ==========================================
# 3. BUILD PREPROCESSOR 
# ==========================================
numeric_features = ['Age', 'LeadDays', 'DayOfWeek']
categorical_features = ['Gender']

for col in ['Scholarship', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received']:
    if col in X.columns:
        numeric_features.append(col)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) 
    ])

# ==========================================
# 4. TRAIN BALANCED RANDOM FOREST 
# ==========================================
print("--- Training Balanced Random Forest Pipeline ---")

# Your brilliant fix: class_weight='balanced'
rf_balanced = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', # This tells the AI that No-Shows are hyper-important!
    random_state=42, 
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_balanced)
])

rf_pipeline.fit(X_train, y_train)

# ==========================================
# 5. BUSINESS THRESHOLD (45%)
# ==========================================
# Predict probabilities on test set
y_pred_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
print(f"✅ Balanced Model ROC-AUC: {roc_auc_rf:.4f}\n")

# FIXED: Changed from 0.40 to 0.45 to match your comments
BUSINESS_THRESHOLD = 0.45
y_pred_custom = (y_pred_proba_rf >= BUSINESS_THRESHOLD).astype(int)

print(f"--- Evaluation at {BUSINESS_THRESHOLD * 100}% Business Threshold ---")
print(classification_report(y_test, y_pred_custom))

# ==========================================
# 6. SAVE THE PIPELINE & THRESHOLD
# ==========================================
os.makedirs('D:\\Desktop\\BootCamp_Hackathon\\models', exist_ok=True)
model_path = 'D:\\Desktop\\BootCamp_Hackathon\\models\\noshow_model_rf.joblib'
joblib.dump(rf_pipeline, model_path)

threshold_path = 'D:\\Desktop\\BootCamp_Hackathon\\models\\threshold.txt'
with open(threshold_path, 'w') as f:
    f.write(str(BUSINESS_THRESHOLD))

print(f"\n✅ Production-Ready Balanced Pipeline saved to {model_path}")
print(f"🎯 Threshold set to {BUSINESS_THRESHOLD * 100}% with class_weight='balanced'")

Loading cleaned data...
Training on 88417 rows, Testing on 22105 rows...

--- Training Balanced Random Forest Pipeline ---
✅ Balanced Model ROC-AUC: 0.7270

--- Evaluation at 45.0% Business Threshold ---
              precision    recall  f1-score   support

           0       0.95      0.43      0.59     17642
           1       0.29      0.91      0.44      4463

    accuracy                           0.53     22105
   macro avg       0.62      0.67      0.52     22105
weighted avg       0.82      0.53      0.56     22105


✅ Production-Ready Balanced Pipeline saved to D:\Desktop\BootCamp_Hackathon\models\noshow_model_rf.joblib
🎯 Threshold set to 45.0% with class_weight='balanced'
